# Module 5.3 — Market Making & Arbitrage
## Liquidity, Dislocations, and the Race to Fair Value

---

### On Providing Liquidity vs. Exploiting Gaps

**Market makers** stand in the middle: they **quote** bid and ask, earn the **spread** when flow is balanced, and absorb **inventory risk** when it is not. **Arbitrageurs** stand at the edges: they hunt **inconsistencies**—between related instruments, venues, or conversion paths—until prices **line up** (net of fees, funding, and frictions). In practice the lines blur: a desk may both **make** and **lean** on dislocations; **statistical arbitrage** from Module 5.2 is a cousin of pure arb, with **model risk** instead of a locked identity.

This notebook is a **compact map**: spread capture with **inventory skew**, **triangular FX** consistency, **cross-exchange** detection after costs, **index** futures vs. fair value, **crypto** frictions, and **latency** as a first-order constraint in modern markets.

---

### Learning Objectives

1. **Explain** how bid–ask quoting earns spread and why **inventory** forces **skew** or **limits**
2. **Simulate** a minimal **market-making** process with fill risk and inventory caps
3. **Relate** **statistical arbitrage** to deterministic **arb** (when the hedge is approximate vs. exact)
4. **Detect** **triangular** FX opportunities from three pairwise quotes (before costs)
5. **Build** a **cross-exchange** arbitrage scanner with **fees** and **minimum edge**
6. **Compute** **index futures fair value** and a **basis** signal for cash-and-carry intuition
7. **Describe** **crypto** cross-venue gaps and **latency arbitrage** in operational terms

---

### Module Contents

1. **Bid–ask capture & inventory** — Spread, adverse selection, inventory skew
2. **Statistical arbitrage (bridge)** — From Module 5.2 to “hard” arb
3. **Triangular FX** — Cycle consistency and profit factor
4. **Cross-exchange detector** — Two venues, fees, transferable asset
5. **Index arbitrage** — Fair futures value and basis scanner
6. **Crypto opportunities** — Settlement, transfer, and fragmentation
7. **Latency arbitrage** — Stale quotes and speed asymmetry (concept + toy)

---

*"The spread pays you until informed flow teaches you the fair price."*


In [ ]:
# ========================================
# Setup
# ========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Dict, List, Tuple
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams.update({"figure.figsize": (14, 6), "axes.titlesize": 15, "figure.dpi": 100})

COLORS = {
    "mid": "#1565C0",
    "bid": "#2E7D32",
    "ask": "#C62828",
    "inv": "#6A1B9A",
    "pnl": "#00838F",
    "edge": "#E65100",
}

print("=" * 78)
print(" " * 8 + "MODULE 5.3: MARKET MAKING & ARBITRAGE")
print("=" * 78)
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


---

## 1. Bid–Ask Spread Capture & Inventory Management

### Why quants need this

**Revenue** from market making is roughly **spread × volume** minus **adverse selection** (when your quote is picked off by informed traders) and **inventory risk** (mark-to-market swings while you hold a position). Quants model **optimal quotes** (e.g. Avellaneda–Stoikov-style skew and width) and **limits** so inventory does not dominate the PnL. Without inventory awareness, a simple “always quote symmetrically” rule **bleeds** in trending or one-sided flow.

### Sketch

Let $m_t$ be mid, half-spread $s$. Naive quotes: bid $= m_t - s$, ask $= m_t + s$. With inventory $q_t$, a common **linear skew** pushes quotes in the direction that **reduces** $|q|$—e.g. shift both quotes down when long so **selling** (hits on the bid from your perspective—actually you want to attract buyers when short; convention varies by implementation). Here we use: **widen** or **disable** the side that **adds** to inventory beyond a cap, and add a small **gamma** skew to mid for inventory mean-reversion.


In [ ]:
def simulate_market_making(
    n_steps: int = 800,
    mid0: float = 100.0,
    daily_vol: float = 0.20,
    half_spread: float = 0.08,
    gamma_skew: float = 0.02,
    fill_prob: float = 0.22,
    max_inventory: int = 15,
    seed: int = 42,
) -> pd.DataFrame:
    # Toy MM: log-normal mid; with prob fill_prob a trade may occur (random buy/sell flow).
    rng = np.random.default_rng(seed)
    dt = 1 / 252
    sig_sqrt_dt = daily_vol * np.sqrt(dt)

    mid = np.zeros(n_steps)
    inv = np.zeros(n_steps, dtype=np.int32)
    cash = np.zeros(n_steps)
    mid[0] = mid0

    for t in range(1, n_steps):
        mid[t] = mid[t - 1] * np.exp(-0.5 * daily_vol**2 * dt + sig_sqrt_dt * rng.standard_normal())
        q = int(inv[t - 1])
        center = mid[t] - gamma_skew * q
        bid = center - half_spread
        ask = center + half_spread

        can_buy = q < max_inventory
        can_sell = q > -max_inventory

        if rng.random() >= fill_prob:
            inv[t] = q
            cash[t] = cash[t - 1]
            continue

        # 0: aggressor sells -> we lift bid (inventory up); 1: aggressor buys -> we hit ask (inventory down)
        side = rng.integers(0, 2)
        if side == 0 and can_buy:
            inv[t] = q + 1
            cash[t] = cash[t - 1] - bid
        elif side == 1 and can_sell:
            inv[t] = q - 1
            cash[t] = cash[t - 1] + ask
        else:
            inv[t] = q
            cash[t] = cash[t - 1]

    df = pd.DataFrame(
        {
            "mid": mid,
            "inventory": inv,
            "cash": cash,
            "mtm": cash + inv.astype(float) * mid,
        }
    )
    df["pnl_change"] = df["mtm"].diff().fillna(0)
    return df


mm = simulate_market_making()
print("Market-making toy run (last 5 rows):")
print(mm.tail())

fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
ax[0].plot(mm.index, mm["mid"], color=COLORS["mid"], lw=1)
ax[0].set_title("Mid price (synthetic)")
ax[1].step(mm.index, mm["inventory"], where="post", color=COLORS["inv"], lw=1)
ax[1].axhline(0, color="gray", lw=0.6)
ax[1].set_title("Inventory (lots)")
ax[2].plot(mm.index, mm["mtm"], color=COLORS["pnl"], lw=1.2)
ax[2].set_title("Mark-to-market wealth (cash + inventory × mid)")
plt.tight_layout()
plt.show()


---

## 2. Statistical Arbitrage vs. “Hard” Arbitrage

### Why quants need this

**Pure arbitrage** (same cash flows, zero net investment, positive payoff) is rare in liquid markets; **quasi-arb** appears after **fees**, **borrow**, **settlement**, and **model error**. **Statistical arbitrage** (Module 5.2) treats the **spread** as mean-reverting **on average**—profitable until the relationship **breaks**. Quants need the vocabulary to separate **identity-based** locks (FX triangle, index parity) from **estimated** edges (pairs, factors).

### One-line distinction

| Idea | Hedge | Risk |
|------|--------|------|
| Hard arb | Replicating portfolio / conversion cycle | Execution, fees, credit |
| Stat arb | Regression / cointegration | Model, regime shift |


---

## 3. Triangular Arbitrage in FX

### Why quants need this

Given three **consistent** rates, converting **USD → EUR → GBP → USD** should return **one** (ignoring costs). If the **implied cross** disagrees with the **quoted cross**, a **triangle** may exist. Real desks add **bid–ask**, **latency**, and **credit limits**; the exercise below uses **mid** quotes to show the **algebra**.

### Convention used here

All `rate[from][to]` = units of **to** received per **1** unit of **from** at mid. Then a cycle USD→EUR→GBP→USD multiplies three factors; product $> 1$ suggests **gross** arb (before costs).


In [ ]:
def triangular_fx_factor(rates: Dict[str, Dict[str, float]], cycle: List[str]) -> float:
    # Multiply rates along cycle[0]->cycle[1]->...->cycle[0].
    f = 1.0
    for i in range(len(cycle) - 1):
        a, b = cycle[i], cycle[i + 1]
        f *= rates[a][b]
    return f


def scan_triangular(
    rates: Dict[str, Dict[str, float]], currencies: List[str]
) -> pd.DataFrame:
    # All directed 3-cycles (for 3 currencies, 2 directions).
    rows = []
    a, b, c = currencies
    cycles = [
        [a, b, c, a],
        [a, c, b, a],
    ]
    for cy in cycles:
        fac = triangular_fx_factor(rates, cy)
        rows.append({"cycle": " → ".join(cy), "factor": fac, "bps_gross": (fac - 1) * 1e4})
    return pd.DataFrame(rows)


# Consistent triangle (approximately)
rates_ok = {
    "USD": {"EUR": 0.92, "GBP": 0.79},
    "EUR": {"USD": 1 / 0.92, "GBP": 0.86},
    "GBP": {"USD": 1 / 0.79, "EUR": 1 / 0.86},
}
df_ok = scan_triangular(rates_ok, ["USD", "EUR", "GBP"])
print("Consistent mids (factor should be ~1):")
print(df_ok.to_string(index=False))

# Introduce a stale EUR/GBP — dislocation
rates_bad = {k: {k2: v2 for k2, v2 in v.items()} for k, v in rates_ok.items()}
rates_bad["EUR"]["GBP"] = 0.90
rates_bad["GBP"]["EUR"] = 1 / 0.90

df_bad = scan_triangular(rates_bad, ["USD", "EUR", "GBP"])
print("\nWith mispriced EUR/GBP cross:")
print(df_bad.to_string(index=False))


---

## 4. Cross-Exchange Arbitrage Detector

### Why quants need this

The same asset may trade at **different prices** on two **venues**. **Gross** edge is easy; **net** edge must subtract **fees**, **slippage**, and (for crypto) **withdrawal time** and **chain risk**. The scanner below flags moments when **buy low / sell high** exceeds a **minimum bps** after **round-trip** costs.

### Assumptions (toy)

- **Instant** transfer or pre-funded balances on both venues (ignores latency and inventory placement).
- **Symmetric** fee in bps on each leg.


In [ ]:
@dataclass
class VenueQuote:
    name: str
    bid: float
    ask: float
    fee_bps: float = 5.0


def cross_exchange_edge(buy_at: VenueQuote, sell_at: VenueQuote) -> Dict[str, float]:
    # Buy at buy_at (lift ask), sell at sell_at (hit bid); fee each leg in bps.
    buy_px = buy_at.ask * (1 + buy_at.fee_bps / 1e4)
    sell_px = sell_at.bid * (1 - sell_at.fee_bps / 1e4)
    edge_per_unit = sell_px - buy_px
    edge_bps = (edge_per_unit / buy_px) * 1e4 if buy_px > 0 else np.nan
    return {
        "buy_venue": buy_at.name,
        "sell_venue": sell_at.name,
        "edge_per_unit": edge_per_unit,
        "edge_bps": edge_bps,
    }


def scan_all_pairs(venues: List[VenueQuote]) -> pd.DataFrame:
    rows = []
    for i, v_buy in enumerate(venues):
        for j, v_sell in enumerate(venues):
            if i == j:
                continue
            rows.append(cross_exchange_edge(v_buy, v_sell))
    return pd.DataFrame(rows)


# Synthetic: two exchanges, transient gap
v_a = VenueQuote("Exchange A", bid=99.85, ask=100.05, fee_bps=8)
v_b = VenueQuote("Exchange B", bid=100.20, ask=100.40, fee_bps=8)

scan_df = scan_all_pairs([v_a, v_b])
print("Cross-venue scan (all directed buy→sell):")
print(scan_df.to_string(index=False))

best = scan_df.loc[scan_df["edge_per_unit"].idxmax()]
print(f"\nBest gross direction: buy {best['buy_venue']}, sell {best['sell_venue']}, edge_bps={best['edge_bps']:.2f}")

# Time series: random mids + occasional gap
rng = np.random.default_rng(1)
n = 400
mid = 100 + np.cumsum(rng.standard_normal(n) * 0.03)
spread_a = 0.10
spread_b = 0.12
# B is sometimes cheap on ask vs A bid
gap = np.zeros(n)
gap[150:200] = 0.35
ask_a = mid + spread_a / 2
bid_a = mid - spread_a / 2
ask_b = mid + spread_b / 2 - gap
bid_b = mid - spread_b / 2 - gap * 0.5

fee = 10
edges = []
for t in range(n):
    va = VenueQuote("A", bid_a[t], ask_a[t], fee)
    vb = VenueQuote("B", bid_b[t], ask_b[t], fee)
    e1 = cross_exchange_edge(va, vb)["edge_bps"]
    e2 = cross_exchange_edge(vb, va)["edge_bps"]
    edges.append(max(e1, e2))

ts = pd.Series(edges, name="max_edge_bps")
signals = ts[ts > 15]
print(f"\nTime-series scan: {len(signals)} bars with max edge > 15 bps (toy)")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts.index, ts.values, color=COLORS["edge"], lw=1)
ax.axhline(15, color="gray", ls="--", label="min edge threshold")
ax.set_title("Cross-exchange max directed edge (bps) — toy series")
ax.legend()
plt.tight_layout()
plt.show()


---

## 5. Index Arbitrage & Basis Scanner

### Why quants need this

**Cash-and-carry** links **spot index** $S$, **risk-free** $r$, **dividend yield** $q$, and **future** $F$ with maturity $T$:
$$F_{\text{fair}} \approx S \, e^{(r - q) T}.$$
If **listed futures** trade **rich** vs. fair, an **index arbitrageur** may **sell** futures and **buy** the **replicating basket** (simplified here as “the index”), subject to **transaction costs**, **basket weight** drift, and **corporate actions**. The scanner flags **basis** in **bps** vs. fair.

### Practice note

Real **index arb** is an **operations** and **financing** problem as much as a formula.


In [ ]:
def fair_future_price(spot: float, r: float, q: float, T_years: float) -> float:
    return spot * np.exp((r - q) * T_years)


def index_basis_bps(spot: float, future: float, r: float, q: float, T_years: float) -> float:
    f_fair = fair_future_price(spot, r, q, T_years)
    return (future - f_fair) / f_fair * 1e4


def scan_index_futures(rows: List[dict]) -> pd.DataFrame:
    out = []
    for row in rows:
        bps = index_basis_bps(row["spot"], row["future"], row["r"], row["q"], row["T"])
        out.append({**row, "fair_future": fair_future_price(row["spot"], row["r"], row["q"], row["T"]), "basis_bps": bps})
    return pd.DataFrame(out)


universe = [
    {"name": "US broad", "spot": 5200.0, "future": 5235.0, "r": 0.045, "q": 0.015, "T": 0.25},
    {"name": "US broad (rich)", "spot": 5200.0, "future": 5280.0, "r": 0.045, "q": 0.015, "T": 0.25},
    {"name": "US broad (cheap)", "spot": 5200.0, "future": 5210.0, "r": 0.045, "q": 0.015, "T": 0.25},
]

scan_idx = scan_index_futures(universe)
print("Index futures basis vs. fair value:")
print(scan_idx[["name", "spot", "future", "fair_future", "basis_bps"]].to_string(index=False))

rich = scan_idx.loc[scan_idx["basis_bps"].idxmax()]
print(f"\nRichest contract vs. model: {rich['name']}, basis {rich['basis_bps']:.1f} bps")


---

## 6. Crypto Arbitrage Opportunities (Frictions)

### Why quants need this

**Crypto** markets are **fragmented**: hundreds of pairs and venues, **24/7** trading, and **on-chain** settlement. **Apparent** price gaps are common; **realizable** arb requires:

- **Withdrawal/deposit** availability and **confirmation** lags
- **Wallet** and **chain** risk
- **Funding** on perpetuals vs. spot
- **Regulatory** and **counterparty** constraints

Quants treat crypto “arb” as **logistics-constrained** edge, not a spreadsheet alone.


### Checklist (conceptual)

| Friction | Effect on arb |
|----------|----------------|
| Transfer time | Price can move before coins arrive |
| Fees (trade + chain) | Erodes small edges |
| Stale order book | Quote already gone when you click |
| Perp funding | Carrying a hedge has a carry |

No separate code cell is required here—the **cross-exchange** machinery above is the same **kernel** with different **latency and funding** assumptions.


---

## 7. Latency Arbitrage (Concepts + Toy)

### Why quants need this

If some participants see **stale** quotes while others see a **new** mid, **speed** can monetize **mechanical** dislocations before others **cancel** or **reprice**. This is **controversial** and **regulated** in some venues; the point for learners is **information time** and **queue position**, not a recipe.

### Toy

Two traders: **slow** observes mid with lag $L$; **fast** observes **current** mid. When volatility jumps, stale quotes can be **picked off**. We simulate a random walk mid and count periods where a **fixed** stale quote would be **arbitraged** by an informed counterparty (simplified).


In [ ]:
def latency_arb_toy(
    n: int = 2000,
    lag: int = 5,
    half_spread: float = 0.15,
    vol: float = 0.5,
    seed: int = 0,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dt = 1 / 252
    z = rng.standard_normal(n)
    mid = 100 * np.exp(np.cumsum(vol * np.sqrt(dt) * z) - 0.5 * vol**2 * dt * np.arange(n))
    stale = pd.Series(mid).shift(lag).values
    # If stale bid/ask were left fixed at t-lag while mid moved: ask_pickoff = mid > stale + half_spread (someone buys cheap from us — we sold too cheap)
    # Simplified flag: mid moved away from stale center by more than 2 * half_spread
    displacement = np.abs(mid - stale)
    pickable = displacement > 2 * half_spread
    return pd.DataFrame({"mid": mid, "stale": stale, "pickable": pickable})


lat = latency_arb_toy()
print(f"Fraction of steps with large stale vs. mid dislocation: {lat['pickable'].mean():.2%}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(lat.index, lat["mid"], label="Mid (fast)", color=COLORS["mid"], lw=0.8)
ax.plot(lat.index, lat["stale"], label=f"Stale view (lag={5})", color=COLORS["edge"], lw=0.8, alpha=0.8)
ax.set_title("Latency toy: current mid vs. lagged observation")
ax.legend()
plt.tight_layout()
plt.show()

print("\n" + "=" * 78)
print("MODULE 5.3 — SUMMARY")
print("=" * 78)
print("MM: spread minus inventory and adverse selection; inventory skew matters.")
print("FX triangle: multiply conversion factors; >1 gross profit before costs.")
print("Cross-venue: always net fees; crypto adds transfer and funding reality.")
print("Index: fair F from S, r, q, T; basis drives arb desks in production.")
print("Latency: stale quotes vs. fast feed — policy and microstructure, not just math.")
print("=" * 78)
